# Funding Rates — Raw Data Explorer

In [ ]:
import pandas as pd

btc = pd.read_parquet('../data/btc_funding_rates.parquet')
eth = pd.read_parquet('../data/eth_funding_rates.parquet')

## Schema

In [ ]:
btc.dtypes

## First rows

In [ ]:
btc.head(10)

## Last rows

In [ ]:
btc.tail(10)

## Coverage & gaps

Funding settles every 8h — there should be exactly 3 rows per day.

In [9]:
print(f"BTC rows: {len(btc)}")
print(f"ETH rows: {len(eth)}")
print(f"Date range: {btc['timestamp'].min()} → {btc['timestamp'].max()}")

# Check for gaps — allow ±10s tolerance for ms-level rounding in source data
gaps = btc['timestamp'].diff().dropna()
tolerance = pd.Timedelta('10s')
unusual = gaps[(gaps - pd.Timedelta('8h')).abs() > tolerance]
print(f"\nRows where gap is not ~8h (±10s): {len(unusual)}")
unusual

BTC rows: 6846
ETH rows: 6846
Date range: 2020-01-01 00:00:00+00:00 → 2026-03-31 16:00:00+00:00

Rows where gap is not ~8h (±10s): 0


Series([], Name: timestamp, dtype: timedelta64[ns])

## Distribution

In [ ]:
btc['funding_rate'].describe()

## Sample of extreme values

In [ ]:
print("Highest funding rates:")
display(btc.nlargest(5, 'funding_rate'))

print("\nLowest funding rates:")
display(btc.nsmallest(5, 'funding_rate'))

## Monthly averages — BTC vs ETH

In [ ]:
btc_monthly = btc.set_index('timestamp').resample('ME')['funding_rate'].mean().rename('btc')
eth_monthly = eth.set_index('timestamp').resample('ME')['funding_rate'].mean().rename('eth')
pd.concat([btc_monthly, eth_monthly], axis=1)